In [ ]:
import os
import sys
import argparse
import pandas as pd
import torch
import anndata as ad
from tqdm import tqdm

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from DeepRUOT.losses import OT_loss1
from DeepRUOT.utils import (
    generate_steps, load_and_merge_config,
    SchrodingerBridgeConditionalFlowMatcher,
    generate_state_trajectory, get_batch, get_batch_size
)
from DeepRUOT.train import train_un1_reduce, train_all
from DeepRUOT.models import FNet_interaction, scoreNet2
from DeepRUOT.constants import DATA_DIR, RES_DIR
from DeepRUOT.exp import setup_exp

### Load config

In [ ]:
config_path = '../config/mosta_config.yaml'

# Load and merge configuration
config = load_and_merge_config(config_path)

### Load data and model

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, config['data']['file_path']))
df = df.iloc[:, :config['data']['dim'] + 1]
#df = df[df.iloc[:,1] > 0.4]
device = torch.device('cpu')
exp_dir, logger = setup_exp(
            RES_DIR, 
            config, 
            config['exp']['name']
        )
dim = config['data']['dim']

In [ ]:
model_config = config['model']
        
f_net = FNet_interaction(
            in_out_dim=model_config['in_out_dim'],
            hidden_dim=model_config['hidden_dim'],
            n_hiddens=model_config['n_hiddens'],
            activation=model_config['activation'],
            use_spatial = True, 
            num_heads = 8,
            thre = 0.06,
            num_layers = 1,

        ).to(device)

sf2m_score_model = scoreNet2(
    in_out_dim=model_config['in_out_dim'],
    hidden_dim=model_config['score_hidden_dim'],
    activation=model_config['activation']
).float().to(device)

In [ ]:
f_net.load_state_dict(torch.load(os.path.join(exp_dir, 'model_final'),map_location=torch.device('cpu')))
f_net.to(device)
sf2m_score_model.load_state_dict(torch.load(os.path.join(exp_dir, 'score_model'),map_location=torch.device('cpu')))
sf2m_score_model.to(device)

### Plot Growth

In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import scanpy as sc
import os

# ==========================================
# 1. Nature Methods Style Configuration
# ==========================================
plt.rcParams.update({
    'font.size': 8,
    'axes.titlesize': 10,
    'axes.labelsize': 8,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica'],
    'pdf.fonttype': 42,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': False,
    'axes.spines.bottom': False,
    'figure.dpi': 300
})

# -------------------------------------------------------
# 工具函数：解析时间 & 空间归一化
# -------------------------------------------------------
def parse_time_from_filename(path):
    try:
        fname = os.path.basename(path)
        segment = fname.split("_")[1].replace("adata_", "").replace("t", "").split(".")[0]
        segment = segment.replace("n", "-").replace("p", ".")
        return float(segment)
    except:
        return 0.0

def normalize_spatial_coords(adata):
    """
    将空间坐标(Spatial)统一缩放到 [0,1]，保持长宽比。
    解决不同时间点Y轴范围不一致的问题。
    """
    coords = adata.obsm["spatial"].copy()
    
    # 简单的 IQR 过滤，防止离群点压缩图像
    y = coords[:, 1]
    Q1, Q3 = np.percentile(y, [25, 75])
    IQR = Q3 - Q1
    mask = (y >= (Q1 - 1.5*IQR)) & (y <= (Q3 + 1.5*IQR))
    adata_filtered = adata[mask].copy()
    coords = adata_filtered.obsm["spatial"]
    
    # Min-Max Scaling to [0,1]
    min_vals = coords.min(axis=0)
    max_vals = coords.max(axis=0)
    scale = (max_vals - min_vals).max() 
    
    adata_filtered.obsm["spatial"] = (coords - min_vals) / scale
    return adata_filtered

# -------------------------------------------------------
# 主流程
# -------------------------------------------------------
def plot_g_nature_style(
    adata_paths, 
    f_net, 
    dim, 
    celltype_key='annotation',  # 新增：列名
    target_celltype='Brain',    # 新增：目标细胞类型
    device="cpu", 
    exp_dir="./"
):
    
    data_by_time = {} 
    
    print(f"--- 1. Processing (Filter: {celltype_key}={target_celltype}) ---")
    
    # === Step 1: 读取、过滤、计算 ===
    for path in adata_paths:
        delta = parse_time_from_filename(path)
        time_value = 12.5 + delta
        label = f"E{time_value:.2f}"
        
        # 1. 读取数据
        adata = sc.read(path)
        
        # 2. 细胞类型过滤 (Cell Type Filtering)
        if celltype_key in adata.obs:
            if target_celltype is not None:
                n_before = adata.shape[0]
                adata = adata[adata.obs[celltype_key] == target_celltype].copy()
                n_after = adata.shape[0]
                if n_after == 0:
                    print(f"Warning: No cells found for {target_celltype} in {label}, skipping.")
                    continue
        else:
            print(f"Warning: {celltype_key} not in adata.obs, using all cells.")

        # 3. Y轴翻转逻辑 (Flip Y-axis for specific timepoints)
        # 注意：在 Normalize 之前翻转
        if np.isclose(time_value, 12.5) or np.isclose(time_value, 15.5):
            print(f"  -> Flipping Y-axis for {label}")
            adata.obsm["spatial"][:, 1] *= -1

        # 4. 空间坐标归一化
        adata = normalize_spatial_coords(adata)
        
        # 5. 模型预测
        new_data = adata.X[:, :dim]
        if hasattr(new_data, "toarray"): new_data = new_data.toarray()
            
        data_tensor = torch.tensor(new_data, dtype=torch.float32).to(device)
        t_tensor = torch.full((data_tensor.shape[0], 1), time_value, dtype=torch.float32).to(device)
        
        f_net.eval()
        with torch.no_grad():
            _, g, _, _ = f_net(t_tensor, data_tensor)
        
        g_numpy = g.detach().cpu().numpy().squeeze()
        
        data_by_time[label] = {
            'x': adata.obsm["spatial"][:, 0],
            'y': adata.obsm["spatial"][:, 1],
            'g_raw': g_numpy # 存储原始值，后面统一处理
        }

    # === Step 2: 数学重构 G 值 (Robust Min-Max Scaling) ===
    print("--- 2. Normalizing G to [0, 1] interval ---")
    
    # 收集所有原始值
    all_g_values = np.concatenate([d['g_raw'] for d in data_by_time.values()])
    
    # 计算稳健的边界 (5% - 95%) 
    # 使用 percentile 而不是 min/max 是为了防止极个别噪点拉伸整个色阶
    g_min = np.percentile(all_g_values, 5)
    g_max = np.percentile(all_g_values, 95)
    
    print(f"Global Raw Stats -> Min: {all_g_values.min():.4f}, Max: {all_g_values.max():.4f}")
    print(f" robust range (5%-95%): [{g_min:.4f}, {g_max:.4f}]")
    
    # 更新字典中的 g 值
    for label in data_by_time:
        g_raw = data_by_time[label]['g_raw']
        
        # 核心数学变换：(x - min) / (max - min)
        # 并截断到 [0, 1] 之间
        g_norm = (g_raw - g_min) / (g_max - g_min)
        g_norm = np.clip(g_norm, 0, 1)
        
        data_by_time[label]['g_final'] = g_norm

    # === Step 3: 绘图 ===
    n = len(data_by_time)
    fig_width = min(n * 6, 14.4)
    fig, axes = plt.subplots(1, n, figsize=(fig_width, 6.0), dpi=300)
    if n == 1: axes = [axes]
    
    # 此时数据已经在 0-1 之间，直接用 0-1 的 norm
    norm = plt.Normalize(vmin=0, vmax=1)
    cmap = plt.cm.RdYlBu_r # 0(蓝/低) -> 1(红/高)

    sorted_labels = sorted(data_by_time.keys(), key=lambda x: float(x[1:])) # 按时间排序 E12.5...
    
    for ax, label in zip(axes, sorted_labels):
        content = data_by_time[label]
        
        ax.scatter(
            content['x'], content['y'], 
            c=content['g_final'], 
            s=0.4, 
            cmap=cmap, 
            norm=norm, 
            linewidths=0, 
            alpha=1.0
        )
        
        ax.set_title(label, pad=4)
        ax.axis('off')
        ax.set_aspect('equal')

    # === Step 4: Colorbar ===
    cax = fig.add_axes([0.92, 0.3, 0.015, 0.4])
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cax)
    
    cbar.set_ticks([0, 0.5, 1])
    # 这里可以使用 Relative 标签，或者显示真实的 Raw 值
    cbar.set_ticklabels(['0', '0.5', '1']) 
    cbar.set_label("Growth Rate", rotation=270, labelpad=10)
    cbar.outline.set_linewidth(0.5)
    cbar.ax.tick_params(length=2, width=0.5)

    save_path = os.path.join(exp_dir, "g_values_brain_normalized.pdf")
    plt.savefig(save_path, bbox_inches='tight', pad_inches=0.1)
    print(f"\nSaved figure to: {save_path}\n")
    plt.show()

# 调用示例
# plot_g_nature_style(adata_paths, f_net, dim, celltype_key='annotation', target_celltype='Brain', device=device)

In [ ]:
# 定义文件路径列表
adata_paths = [
    '/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/results/mosta_interaction_1017_tiaocan/adata_tn2p000.h5ad',
    '/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/results/mosta_interaction_1017_tiaocan/adata_tn1p000.h5ad',
    '/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/results/mosta_interaction_1017_tiaocan/adata_t0p250.h5ad',
    '/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/results/mosta_interaction_1017_tiaocan/adata_t1p250.h5ad',
    '/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/results/mosta_interaction_1017_tiaocan/adata_t2p500.h5ad',
    '/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/results/mosta_interaction_1017_tiaocan/adata_t3p500.h5ad',
    '/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/results/mosta_interaction_1017_tiaocan/adata_t4p000.h5ad',
    '/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/results/mosta_interaction_1017_tiaocan/adata_t5p000.h5ad',
    '/lustre/home/2200012126/spatial_data/CytoBridge-ST-1104/results/mosta_interaction_1017_tiaocan/adata_t6p000.h5ad'
]

In [ ]:
plot_g_nature_style(adata_paths, f_net, dim, celltype_key='annotation', target_celltype='Brain', device=device)